# PDD Mobility Scanner — Trail Object Detection

Detect benches, chairs, and other trailside amenities in body-camera photos using YOLOv8, match detections to GPS coordinates from a companion sensor CSV, and export a CSV ready for map visualization.

Pipeline:
1. Upload `.jpg` images and `trail_data.csv`
2. Run **YOLOv8** object detection on every image
3. Match each image to its GPS waypoint from the CSV
4. Export `trail_detections.csv` with detection + location data

**GPU runtime recommended** (Runtime → Change runtime type → T4 GPU) but CPU works too — inference will just be slower.

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics pandas pillow

## Step 2: Upload images and CSV

In [ ]:
from google.colab import files

uploaded = files.upload()

# Separate images from CSV
csv_files = [f for f in sorted(uploaded.keys()) if f.lower().endswith('.csv')]
image_files = [f for f in sorted(uploaded.keys()) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"\nImages: {len(image_files)}")
print(f"CSV:    {len(csv_files)} — {csv_files}")

## Step 3: Parse GPS coordinates from CSV

The CSV has multiple rows per waypoint (25Hz sampling). We average `lat`/`lon` per waypoint to get one coordinate per image.

In [ ]:
import pandas as pd

df_trail = pd.read_csv(csv_files[0])
print(f"Loaded {csv_files[0]}: {len(df_trail)} rows, {df_trail['wp'].nunique()} waypoints")
print(f"Columns: {list(df_trail.columns)}")

# Handle both 'lon' and 'lng' column naming conventions
lon_col = 'lon' if 'lon' in df_trail.columns else 'lng'

# Average lat/lon per waypoint
wp_coords = df_trail.groupby('wp').agg(
    lat=('lat', 'mean'),
    lon=(lon_col, 'mean')
).reset_index()

print(f"\n{len(wp_coords)} waypoints with averaged coordinates")
wp_coords.head()

## Step 4: Load YOLOv8 model

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# COCO classes of interest for trailside amenities
TARGET_CLASSES = {
    13: 'bench',
    56: 'chair',
    11: 'stop sign',
    58: 'potted plant',
}

CONF_THRESHOLD = 0.25

print(f"YOLOv8n loaded")
print(f"Target classes: {TARGET_CLASSES}")
print(f"Confidence threshold: {CONF_THRESHOLD}")

## Step 5: Run detection on all images

In [ ]:
import re
from PIL import Image

detections = []
yolo_results = {}  # store results for visualization

for fname in image_files:
    img = Image.open(fname)
    img_w, img_h = img.size
    img_area = img_w * img_h

    # Run inference
    results = model(fname, conf=CONF_THRESHOLD, verbose=False)
    result = results[0]
    yolo_results[fname] = result

    # Extract waypoint number from filename (img_XXXX.jpg → XXXX)
    match = re.search(r'(\d+)', fname)
    wp_num = int(match.group(1)) if match else None

    # Look up averaged GPS coordinates
    lat, lon = None, None
    if wp_num is not None:
        wp_row = wp_coords[wp_coords['wp'] == wp_num]
        if not wp_row.empty:
            lat = wp_row.iloc[0]['lat']
            lon = wp_row.iloc[0]['lon']

    # Process each detection box
    for box in result.boxes:
        cls_id = int(box.cls[0])
        if cls_id not in TARGET_CLASSES:
            continue

        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        bbox_area = (x2 - x1) * (y2 - y1)
        bbox_area_pct = (bbox_area / img_area) * 100

        detections.append({
            'filename': fname,
            'waypoint': wp_num,
            'lat': lat,
            'lon': lon,
            'class': TARGET_CLASSES[cls_id],
            'confidence': round(conf, 4),
            'bbox_x1': round(x1, 1),
            'bbox_y1': round(y1, 1),
            'bbox_x2': round(x2, 1),
            'bbox_y2': round(y2, 1),
            'bbox_area_pct': round(bbox_area_pct, 2),
        })

    n_det = sum(1 for box in result.boxes if int(box.cls[0]) in TARGET_CLASSES)
    if n_det > 0:
        print(f"{fname}: {n_det} detection(s)")

print(f"\nDone. {len(detections)} total detections across {len(image_files)} images.")

## Step 6: Build output CSV

In [ ]:
df_det = pd.DataFrame(detections, columns=[
    'filename', 'waypoint', 'lat', 'lon', 'class', 'confidence',
    'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'bbox_area_pct'
])

df_det

## Step 7: Summary

In [ ]:
images_with_det = df_det['filename'].nunique() if len(df_det) > 0 else 0

print(f"Total images processed: {len(image_files)}")
print(f"Images with detections: {images_with_det}")
print(f"Total detections:       {len(df_det)}")
print()
if len(df_det) > 0:
    print("Detections per class:")
    for cls_name, count in df_det['class'].value_counts().items():
        print(f"  {cls_name}: {count}")

## Step 8: Visualize detections

Grid of the first 20 images that have detections, with bounding boxes drawn.

In [ ]:
import matplotlib.pyplot as plt

# Get filenames that have at least one target detection
files_with_det = [f for f in image_files
                  if any(int(box.cls[0]) in TARGET_CLASSES for box in yolo_results[f].boxes)]

show_files = files_with_det[:20]
n = len(show_files)

if n == 0:
    print("No target-class detections to visualize.")
else:
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for idx, fname in enumerate(show_files):
        r, c = divmod(idx, cols)
        ax = axes[r][c]

        # Use ultralytics built-in plotting
        result = yolo_results[fname]
        plotted = result.plot()
        # plotted is BGR numpy array, convert to RGB
        ax.imshow(plotted[:, :, ::-1])
        ax.set_title(fname, fontsize=9)
        ax.axis('off')

    # Hide unused subplots
    for idx in range(n, rows * cols):
        r, c = divmod(idx, cols)
        axes[r][c].axis('off')

    plt.tight_layout()
    plt.show()

## Step 9: Export and download

In [ ]:
output_path = 'trail_detections.csv'
df_det.to_csv(output_path, index=False)
print(f"Saved {output_path} ({len(df_det)} rows)")

files.download(output_path)